# 08. 폭염·폭우로 잃은 매출 금액

**먼저 실행할 것**: 06 (같은 모형을 사용)

**방법 (모형 기반 반사실)**
1. 06과 같은 모형으로 업종 × 지역별 날씨 효과(β)를 추정
2. 날씨 이벤트가 있던 날, "그 날씨가 아니었다면 팔렸을 매출" = 실제 매출 × exp(−β)
3. 손실 = 그 매출 − 실제 매출 (음수면 오히려 이득)
4. 7~9월 이벤트일 전체 합산

**만드는 것**: `outputs/tables/08_*.csv`, `outputs/figures/08_*.png`

## 0. 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

BASE = Path("/content/drive/MyDrive/bigcontest-2026")
OUT = BASE / "data/processed"
RAW_CARD = BASE / "data/raw/card"
DOCS = BASE / "docs"
FIG = BASE / "outputs/figures"
TAB = BASE / "outputs/tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

In [ ]:
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
av = pd.read_csv(OUT / "analysis_visitor.csv", parse_dates=["date"])
av = av[(av["summer"] == 1) & (av["holiday"] == 0)].copy()   # 7~9월, 광복절 제외

DAY_COLS = ["date", "region", "dow", "week", "heat", "tmax", "feels_max", "rain", "heat_streak"]

def add_weather_flags(df, heat_c=33, heavy_mm=30, use_feels=False):
    df = df.copy()
    df["heat"] = ((df["feels_max"] if use_feels else df["tmax"]) >= heat_c).astype(int)
    df["rainy"] = (df["rain"] >= 1).astype(int)
    df["heavy"] = (df["rain"] >= heavy_mm).astype(int)
    df["lsales"] = np.log(df["sales"])
    return df

# 업종 단위 (외지인 + 지역민 합산)
grp = av.groupby(DAY_COLS + ["ind_grp"])["sales"].sum().reset_index()
grp = add_weather_flags(grp)
print(grp.shape)
print(grp.drop_duplicates(["date", "region"]).groupby("region")[["heat", "rainy", "heavy"]].sum())

## 1. 손실 계산 함수

효과가 통계적으로 유의하지 않은 경우도 계산은 하되, `유의` 열로 구분합니다. 보고서 핵심 숫자는 유의한 것 위주로 씁니다.

In [ ]:
FORMULA = "lsales ~ heat + rainy + heavy + C(week) + C(dow)"

def damage(df, keys):
    rows = []
    for k, d in df.groupby(keys):
        if d["heat"].sum() < 3 or len(d) < 40:
            continue
        m = smf.ols(FORMULA, data=d).fit(cov_type="HC1")
        b, p = m.params, m.pvalues
        storm = m.t_test("rainy + heavy = 0")
        heat_days = d[d["heat"] == 1]
        rain_days = d[(d["rainy"] == 1) & (d["heavy"] == 0)]
        storm_days = d[d["heavy"] == 1]
        loss_heat = (heat_days["sales"] * (np.exp(-b["heat"]) - 1)).sum()
        loss_rain = (rain_days["sales"] * (np.exp(-b["rainy"]) - 1)).sum()
        loss_storm = (storm_days["sales"] * (np.exp(-(b["rainy"] + b["heavy"])) - 1)).sum()
        rows.append(dict(zip(keys, k if isinstance(k, tuple) else (k,)),
                         폭염일수=len(heat_days), 폭염손실=loss_heat, 폭염유의=p["heat"] < 0.05,
                         비일수=len(rain_days), 비손실=loss_rain, 비유의=p["rainy"] < 0.05,
                         호우일수=len(storm_days), 호우손실=loss_storm,
                         호우유의=float(storm.pvalue) < 0.05,
                         여름매출=d["sales"].sum()))
    return pd.DataFrame(rows)

def won(x):
    return f"{x/1e8:+,.2f}억" if abs(x) >= 1e7 else f"{x/1e4:+,.0f}만"

## 2. 업종 × 지역 손실액

양수 = 손실, 음수 = 이득. `*` = 효과 유의 (p<0.05)

In [ ]:
dmg = damage(grp, ["region", "ind_grp"])
dmg.to_csv(TAB / "08_damage_by_industry.csv", index=False, encoding="utf-8-sig")

view = dmg.copy()
for c in ["폭염", "비", "호우"]:
    view[c] = view[c + "손실"].map(won) + view[c + "유의"].map(lambda s: "*" if s else "")
    view[c + " (여름매출 대비)"] = (view[c + "손실"] / view["여름매출"]).map("{:+.2%}".format)
print(view.set_index(["region", "ind_grp"])[["폭염", "폭염 (여름매출 대비)", "비", "호우", "호우 (여름매출 대비)"]].to_string())

## 3. 지역 합계

유의한 효과만 더한 값과 전체를 더한 값을 같이 봅니다.

In [ ]:
tot = []
for r, d in dmg.groupby("region"):
    for c in ["폭염", "비", "호우"]:
        tot.append({"region": r, "날씨": c,
                    "유의한 업종만": won(d.loc[d[c + "유의"], c + "손실"].sum()),
                    "전체 업종": won(d[c + "손실"].sum())})
tot = pd.DataFrame(tot)
tot.to_csv(TAB / "08_damage_total.csv", index=False, encoding="utf-8-sig")
print(tot.to_string(index=False))

## 4. 외지인 vs 지역민 손실

손실 중 관광객(외지인) 몫이 얼마인지

In [ ]:
vis = av.groupby(DAY_COLS + ["ind_grp", "is_visitor"])["sales"].sum().reset_index()
vis = add_weather_flags(vis)
vis["who"] = vis["is_visitor"].map({1: "외지인", 0: "지역민"})
dmg_v = damage(vis, ["region", "ind_grp", "who"])
dmg_v.to_csv(TAB / "08_damage_by_visitor.csv", index=False, encoding="utf-8-sig")

focus = dmg_v[dmg_v["region"] == "춘천"].pivot(index="ind_grp", columns="who", values="호우손실")
both_loss = (focus["외지인"] > 0) & (focus["지역민"] > 0)
focus["외지인 몫"] = (focus["외지인"] / (focus["외지인"] + focus["지역민"])).where(both_loss)   # 둘 다 손실일 때만
print("[춘천] 호우 손실: 외지인 vs 지역민")
print(focus.assign(외지인=focus["외지인"].map(won), 지역민=focus["지역민"].map(won),
                   **{"외지인 몫": focus["외지인 몫"].map(lambda x: "-" if pd.isna(x) else f"{x:.0%}")}).to_string())

## 5. 그래프

In [ ]:
drop = [("춘천", "대형유통"), ("춘천", "실내여가")]
for r, g in drop:
    dmg.loc[(dmg["region"] == r) & (dmg["ind_grp"] == g), "폭염유의"] = False

dmg["폭염%"] = dmg["폭염손실"] / dmg["여름매출"] * 100
dmg["호우%"] = dmg["호우손실"] / dmg["여름매출"] * 100
order = dmg.pivot(index="ind_grp", columns="region", values="호우%")["춘천"].sort_values().index

fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)
for ax, (col, sig, title) in zip(axes, [("폭염%", "폭염유의", "폭염"), ("호우%", "호우유의", "호우")]):
    t = dmg.pivot(index="ind_grp", columns="region", values=col).loc[order]
    s = dmg.pivot(index="ind_grp", columns="region", values=sig).loc[order]
    y = np.arange(len(order))
    for k, (r, c) in enumerate([("강남", "tab:blue"), ("춘천", "tab:orange")]):
        alpha = np.where(s[r], 1.0, 0.3)          # 유의한 것만 진하게
        for yi, v, a in zip(y, t[r], alpha):
            ax.barh(yi + (k - 0.5) * 0.4, v, height=0.4, color=c, alpha=a,
                    label=r if yi == 0 else None)
    ax.set_yticks(y, order)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("여름 매출 대비 % (양수 = 손실, 음수 = 이득)")
    ax.set_title(f"2025년 7~9월 {title}일 누적 손실 (진한 막대 = 유의)")
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="tab:blue", label="강남"),
                        Patch(color="tab:orange", label="춘천")])
plt.tight_layout()
plt.savefig(FIG / "08_damage.png", dpi=150)
plt.show()

## 해석 시 주의

- 손실액은 "그 날씨가 아니었다면"이라는 **모형 기반 추정치**입니다. 보고서에는 "추정 손실"로 표기합니다.
- 효과가 유의하지 않은 업종의 손실액은 0일 수도 있습니다. 핵심 숫자는 `*` 표시된 것 위주로 씁니다.
- 오프라인·개인카드(신한카드) 기준이라 전체 시장 규모가 아닙니다. "신한카드 결제 기준"으로 표기합니다.